In [ ]:
import sys
print(sys.executable)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
sns.set()
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

In [ ]:
try:
    import ydata_profiling
except ImportError:
    !{sys.executable} -m pip install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport

from ydata_profiling.utils.cache import cache_zipped_file

In [ ]:
!pip install -q lightautoml

In [ ]:
from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task
from lightautoml.automl.presets.tabular_presets import TabularUtilizedAutoML

In [ ]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [ ]:
df_train

In [ ]:
df_test

In [ ]:
profile = ProfileReport(
        df_train, title="Profile Report of the House Prices Dataset", explorative=True
    )

profile

In [ ]:
profile1 = ProfileReport(
        df_test, title="Profile1 Report of the House Prices Dataset", explorative=True
    )

profile1

In [ ]:
df_train.info()

In [ ]:
df_train.isnull().sum()[:50]

In [ ]:
df_train['SalePrice'].describe()

In [ ]:
%matplotlib inline

In [ ]:
missing = (
    df_train
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing[:20]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_train['SalePrice'], kde=True, ax=ax[0])
ax[0].set_title('Original SalePrice')

sns.histplot(np.log1p(df_train['SalePrice']), kde=True, ax=ax[1])
ax[1].set_title('log1p(SalePrice)')

plt.show()

In [ ]:
train = df_train.copy()
test = df_test.copy()

# feature engineering
train['HouseScore'] = train['OverallQual'] * train['OverallCond']
test['HouseScore'] = test['OverallQual'] * test['OverallCond']

# Общая площадь дома
train['TotalSF'] = (train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF'])
test['TotalSF'] = (test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF'])

# Возраст дома
train['HouseAge'] = (train['YrSold'] - train['YearBuilt'])
test['HouseAge'] = (test['YrSold'] - test['YearBuilt'])

# Сколько прошло после ремонта
train['YearsSinceRemodel'] = (train['YrSold'] - train['YearRemodAdd'])
test['YearsSinceRemodel'] = (test['YrSold'] - test['YearRemodAdd'])

# Есть ли бассейн
train['HasPool'] = (train['PoolArea'] > 0).astype(int)
test['HasPool'] = (test['PoolArea'] > 0).astype(int)

# Общая площадь ванной
train['TotalBath'] = (train['FullBath'] + 0.5 * train['HalfBath'] + train['BsmtFullBath'] + 0.5 * train['BsmtHalfBath'])
test['TotalBath'] = (test['FullBath'] + 0.5 * test['HalfBath'] + test['BsmtFullBath'] + 0.5 * test['BsmtHalfBath'])

X_test = test

In [ ]:
set(X_train.columns) - set(X_test.columns)
set(X_test.columns) - set(X_train.columns)

In [ ]:
y = np.log1p(train['SalePrice'])

X = train.drop(columns=['SalePrice'])

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
train_lama = X_tr.copy()
train_lama['SalePrice'] = y_tr

# задача регрессии
task = Task('reg')

# создаем AutoML-модель
automl = TabularAutoML(
    task=task,
    timeout=600,
    cpu_limit=4,
    reader_params={
        'n_jobs': 4,
        'random_state': 42
    }
)

# роли колонок
roles = {
    'target': 'SalePrice'
}

# обучение
oof_pred = automl.fit_predict(
    train_lama,
    roles=roles,
    verbose=1
)

In [ ]:
# посредстволм AutoML делаем прогноз для всех домов из валидационного набора
pred_log = automl.predict(X_val).data[:, 0]
# делаем обратное преобразование
pred1 = np.expm1(pred_log)
real1 = np.expm1(y_val)

mae1 = mean_absolute_error(real1, pred1)
rmse1 = root_mean_squared_error(real1, pred1)

print(mae1)
print(rmse1)

In [ ]:
automl = TabularUtilizedAutoML(
    task=task,
    timeout=600,
    cpu_limit=4
)

# роли колонок
roles = {
    'target': 'SalePrice'
}

# обучение
oof_pred = automl.fit_predict(
    train_lama,
    roles=roles,
    verbose=1
)

In [ ]:
# посредстволм AutoML делаем прогноз для всех домов из валидационного набора
pred_log = automl.predict(X_val).data[:, 0]
# делаем обратное преобразование
pred2 = np.expm1(pred_log)
real2 = np.expm1(y_val)

mae2 = mean_absolute_error(real2, pred2)
rmse2 = root_mean_squared_error(real2, pred2)

print(mae2)
print(rmse2)

In [ ]:
results = pd.DataFrame({
    'Model': ['TabularAutoML', 'TabularUtilizedAutoML'],
    'MAE': [mae1, mae2],
    'RMSE': [rmse1, rmse2]
})

results

Для решения задачи House Prices была использована библиотека LightAutoML.

После предварительного анализа данных были созданы дополнительные признаки:

* HouseScore;
* TotalSF;
* HouseAge;
* YearsSinceRemodel;
* HasPool;
* TotalBath.

Целевая переменная SalePrice была логарифмирована с помощью функции log1p, что позволило уменьшть асимметрию распределения данных.

В качестве базовой модели использовался TabularAutoML. Для оценки качества данные были разделены на обучающую и валидационную выборки в пропорции 80/20.

Результаты базовой модели:

* MAE = 14886;
* RMSE = 27170.

Далее была протестирована TabularUtilizedAutoML, автоматически перебирающая и строящая расширенный ансамбль моделей.

Результаты улучшенной модели:

* MAE = 14666;
* RMSE = 27436.

Полученная модель показала улучшение по метрике MAE примерно на 1.5%.